In [ ]:
!pip install -q catboost xgboost lightgbm scikit-learn pandas numpy miceforest

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import miceforest as mf
import warnings
warnings.filterwarnings('ignore')

print("=" * 100)
print("✅ 최적화 코드: cat_features 유지 (범주형 정수 변환)")
print("=" * 100)

# =============================================================================
# CELL 0: 데이터 로드
# =============================================================================

print("\n📥 Step 1: 데이터 로드")
print("-" * 100)

df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

print(f"✓ Train: {df_train.shape}")
print(f"✓ Test: {df_test.shape}")

test_ids = df_test['ID'].copy() if 'ID' in df_test.columns else df_test.index.copy()

# =============================================================================
# CELL 1: 기본 전처리
# =============================================================================

print("\n🔧 Step 2: 기본 전처리")
print("-" * 100)

# DI 시술 결측치
di_nan_col = [
    '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
    '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
    '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수',
]

for df in [df_train, df_test]:
    row = df['시술 유형'] == "DI"
    for col in di_nan_col:
        if col in df.columns:
            df.loc[row, col] = df.loc[row, col].fillna(0)

# 횟수 매핑
mapping = {'0회': 0, '1회': 1, '2회': 2, '3회': 3, '4회': 4, '5회': 5, '6회 이상': 6}
count_cols = ["IVF 임신 횟수", "IVF 출산 횟수", "DI 임신 횟수", "DI 출산 횟수",
              "IVF 시술 횟수", "총 임신 횟수", "DI 시술 횟수", "총 시술 횟수",
              "클리닉 내 총 시술 횟수", "총 출산 횟수"]

for df in [df_train, df_test]:
    for col in count_cols:
        if col in df.columns:
            df[col] = df[col].replace(mapping)
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print("✓ 기본 전처리 완료")

# =============================================================================
# CELL 2: MICE 결측치 처리
# =============================================================================

print("\n🔬 Step 3: MICE 결측치 처리")
print("-" * 100)

mice_cols = [
    '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '총 생성 배아 수', '이식된 배아 수',
    '미세주입 배아 이식 수', '저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수',
    '해동 난자 수', '수집된 신선 난자 수', '저장된 신선 난자 수', '혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수', '기증자 정자와 혼합된 난자 수',
    "난자 채취 경과일", "난자 해동 경과일", "배아 이식 경과일", "배아 해동 경과일",
]

mice_cols = [col for col in mice_cols if col in df_train.columns]

print("  MICE 모델 학습 중...", end=" ")
df_train_subset = df_train[mice_cols].copy()
mice_model = mf.ImputationKernel(df_train_subset, random_state=42)
mice_model.mice(iterations=3, n_estimators=30, n_jobs=-1, verbose=False)
print("✓")

print("  Train 결측치 처리...", end=" ")
df_train_imputed = mice_model.complete_data()
df_train[mice_cols] = df_train_imputed[mice_cols]
print("✓")

print("  Test 결측치 처리...", end=" ")
df_test_subset = df_test[mice_cols].copy()
df_test_imputed = mice_model.impute_new_data(df_test_subset).complete_data()
df_test[mice_cols] = df_test_imputed[mice_cols]
print("✓")

print("✓ MICE 완료")

# =============================================================================
# CELL 3: 파생변수
# =============================================================================

print("\n💡 Step 4: 파생변수 생성")
print("-" * 100)

print("  파생변수 생성 중...", end=" ")

def safe_divide(numerator, denominator, default=-1):
    result = np.divide(numerator, denominator, where=(denominator != 0),
                       out=np.full_like(numerator, default, dtype=float))
    result = np.where(np.isinf(result), default, result)
    result = np.where(np.isnan(result), default, result)
    return result

for df in [df_train, df_test]:

    df['배양시간'] = df['배아 이식 경과일'].fillna(0) - df['난자 혼합 경과일'].fillna(0)
    df['배양시간'] = df['배양시간'].clip(-5, 10)

    df['신선만_사용'] = ((df['신선 배아 사용 여부'].fillna(0) == 1) &
                         (df['동결 배아 사용 여부'].fillna(0) == 0)).astype(int)
    df['동결만_사용'] = ((df['동결 배아 사용 여부'].fillna(0) == 1) &
                         (df['신선 배아 사용 여부'].fillna(0) == 0)).astype(int)
    df['혼합_사용'] = ((df['신선 배아 사용 여부'].fillna(0) == 1) &
                       (df['동결 배아 사용 여부'].fillna(0) == 1)).astype(int)

    df['배아전환율'] = safe_divide(
        df['총 생성 배아 수'].fillna(0).values,
        df['수집된 신선 난자 수'].fillna(0).values + 1, default=-1)
    df['배아전환율'] = np.clip(df['배아전환율'], -1, 5)

    df['미세주입성공률'] = safe_divide(
        df['미세주입에서 생성된 배아 수'].fillna(0).values,
        df['미세주입된 난자 수'].fillna(0).values + 1, default=-1)
    df['미세주입성공률'] = np.clip(df['미세주입성공률'], -1, 5)

    df['이전임신성공률'] = safe_divide(
        df['총 임신 횟수'].fillna(0).values,
        df['총 시술 횟수'].fillna(0).values + 1, default=-1)
    df['이전임신성공률'] = np.clip(df['이전임신성공률'], -1, 5)

    df['이전출산성공률'] = safe_divide(
        df['총 출산 횟수'].fillna(0).values,
        df['총 임신 횟수'].fillna(0).values + 1, default=-1)
    df['이전출산성공률'] = np.clip(df['이전출산성공률'], -1, 5)

    df['IVF비율'] = safe_divide(
        df['IVF 시술 횟수'].fillna(0).values,
        df['IVF 시술 횟수'].fillna(0).values + df['DI 시술 횟수'].fillna(0).values + 1,
        default=-1)
    df['IVF비율'] = np.clip(df['IVF비율'], -1, 2)

    df['IVF임신성공률'] = safe_divide(
        df['IVF 임신 횟수'].fillna(0).values,
        df['IVF 시술 횟수'].fillna(0).values + 1, default=-1)
    df['IVF임신성공률'] = np.clip(df['IVF임신성공률'], -1, 5)

    df['DI임신성공률'] = safe_divide(
        df['DI 임신 횟수'].fillna(0).values,
        df['DI 시술 횟수'].fillna(0).values + 1, default=-1)
    df['DI임신성공률'] = np.clip(df['DI임신성공률'], -1, 5)

    df['배아충분도'] = (df['총 생성 배아 수'].fillna(0) -
                       df['이식된 배아 수'].fillna(0)).clip(0, 10)

    df['이상적배양'] = ((df['배아 이식 경과일'].fillna(0) == 3) |
                       (df['배아 이식 경과일'].fillna(0) == 5)).astype(int)

    df['단일배아이식'] = (df['단일 배아 이식 여부'].fillna(0) == 1).astype(int)

    df['경험많음'] = (df['총 시술 횟수'].fillna(0) >= 3).astype(int)
    df['성공경험'] = (df['총 출산 횟수'].fillna(0) >= 1).astype(int)

    df = df.fillna(0)
    df = df.replace([np.inf, -np.inf], 0)

print("✓")
print(f"✓ 18개 파생변수 추가")

# =============================================================================
# CELL 4: 범주형 정수 변환 (최적 방법!)
# =============================================================================

print("\n🔤 Step 5: 범주형을 정수로 변환")
print("-" * 100)

# ID 제거
if 'ID' in df_train.columns:
    df_train = df_train.drop('ID', axis=1)
if 'ID' in df_test.columns:
    df_test = df_test.drop('ID', axis=1)

# 범주형 컬럼 정의 (객체형인 컬럼들)
categorical_cols = [col for col in df_train.columns
                   if df_train[col].dtype == 'object' and col != '임신 성공 여부']

print(f"  범주형 컬럼 감지: {len(categorical_cols)}개")

# LabelEncoder로 범주형을 정수로 변환 (⭐ 안전하고 최적!)
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()

    # Train + Test를 함께 인코딩 (같은 매핑)
    combined = pd.concat([df_train[col], df_test[col]])
    le.fit(combined)

    df_train[col] = le.transform(df_train[col]).astype(int)
    df_test[col] = le.transform(df_test[col]).astype(int)

    le_dict[col] = le

print(f"✓ 범주형을 정수로 변환 완료")

# 나머지 숫자형 처리
for df in [df_train, df_test]:
    for col in df.columns:
        if col != '임신 성공 여부':
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int64')

    df = df.replace([np.inf, -np.inf], 0)

X_train = df_train.drop('임신 성공 여부', axis=1)
y_train = df_train['임신 성공 여부']
X_test = df_test.copy()

# cat_feature_indices 생성 (정수형이므로 안전!)
cat_feature_indices = [i for i, col in enumerate(X_train.columns)
                      if col in categorical_cols]

print(f"✓ cat_features 인덱스: {len(cat_feature_indices)}개")
print(f"✓ 모든 컬럼 정수형으로 변환")

# =============================================================================
# CELL 5: 25-Fold CatBoost (cat_features 유지!)
# =============================================================================

print("\n🤖 Step 6: 25-Fold CatBoost (cat_features 유지)")
print("-" * 100)

params = {
    'bagging_temperature': 0.002,
    'border_count': 167,
    'depth': 7,
    'iterations': 2000,
    'l2_leaf_reg': 5.9,
    'learning_rate': 0.059,
    'loss_function': 'Logloss',
    'od_type': 'Iter',
    'od_wait': 200,
    'random_seed': 2024,
    'random_strength': 0.989,
    'task_type': 'GPU',
    'use_best_model': True,
    'verbose': 0,
    'class_weights': {0: 1.0, 1: 1.005}
}

k_fold = 25
skf = StratifiedKFold(n_splits=k_fold, shuffle=True, random_state=2024)

test_pred_lst = []
eval_metrics = {'roc_auc': [], 'f1': [], 'accuracy': []}

print(f"25-Fold 시작...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    if (fold + 1) % 5 == 0:
        print(f"  [{fold+1}/25] 진행 중...", end="\r")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # ⭐ cat_features 유지 (정수형이므로 안전!)
    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_feature_indices)
    eval_pool = Pool(X_val, label=y_val, cat_features=cat_feature_indices)

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=eval_pool, early_stopping_rounds=200)

    val_pred_proba = model.predict_proba(X_val)[:, 1]
    val_pred = (val_pred_proba > 0.5).astype(int)

    auc = roc_auc_score(y_val, val_pred_proba)
    f1 = f1_score(y_val, val_pred, zero_division=0)
    acc = accuracy_score(y_val, val_pred)

    eval_metrics['roc_auc'].append(auc)
    eval_metrics['f1'].append(f1)
    eval_metrics['accuracy'].append(acc)

    test_pred_lst.append(model.predict_proba(X_test)[:, 1])

print(f"\n✓ 25-Fold 완료")
print(f"  평균 ROC-AUC: {np.mean(eval_metrics['roc_auc']):.5f}")
print(f"  평균 F1-Score: {np.mean(eval_metrics['f1']):.5f}")
print(f"  평균 Accuracy: {np.mean(eval_metrics['accuracy']):.5f}")

# =============================================================================
# CELL 6: 제출 파일 생성
# =============================================================================

print("\n📤 Step 7: 제출 파일 생성")
print("-" * 100)

test_predictions = np.mean(test_pred_lst, axis=0)
test_predictions = np.clip(test_predictions, 0.001, 0.999)

df_sub = pd.read_csv("sample_submission.csv")

if 'ID' in df_sub.columns:
    df_sub['probability'] = test_predictions
else:
    df_sub = pd.DataFrame({
        'ID': test_ids,
        'probability': test_predictions
    })

OUTPUT_NAME = './data/final_submission.csv'
df_sub.to_csv(OUTPUT_NAME, index=False)

print(f"✓ 제출 파일 생성")
print(f"  파일: {OUTPUT_NAME}")
print(f"  샘플: {len(df_sub):,}개")
print(f"  평균: {test_predictions.mean():.6f}")
print(f"  범위: [{test_predictions.min():.6f}, {test_predictions.max():.6f}]")

# =============================================================================
# CELL 7: 최종 결과
# =============================================================================
print("\n📤 Step 7: 제출 파일 생성")
print("-" * 100)

test_predictions = np.mean(test_pred_lst, axis=0)
test_predictions = np.clip(test_predictions, 0.001, 0.999)

# ⭐ Colab 경로로 수정!
df_sub = pd.read_csv("/content/sample_submission.csv")

if 'ID' in df_sub.columns:
    df_sub['probability'] = test_predictions
else:
    df_sub = pd.DataFrame({
        'ID': test_ids,
        'probability': test_predictions
    })

# ⭐ 경로 수정!
OUTPUT_NAME = '/content/final_submission.csv'
df_sub.to_csv(OUTPUT_NAME, index=False)

print(f"✓ 제출 파일 생성")
print(f"  파일: {OUTPUT_NAME}")
print(f"  샘플: {len(df_sub):,}개")
print(f"  평균: {test_predictions.mean():.6f}")
print(f"  범위: [{test_predictions.min():.6f}, {test_predictions.max():.6f}]")

✅ 최적화 코드: cat_features 유지 (범주형 정수 변환)

📥 Step 1: 데이터 로드
----------------------------------------------------------------------------------------------------
✓ Train: (256351, 69)
✓ Test: (90067, 68)

🔧 Step 2: 기본 전처리
----------------------------------------------------------------------------------------------------
✓ 기본 전처리 완료

🔬 Step 3: MICE 결측치 처리
----------------------------------------------------------------------------------------------------
  MICE 모델 학습 중... ✓
  Train 결측치 처리... ✓
  Test 결측치 처리... ✓
✓ MICE 완료

💡 Step 4: 파생변수 생성
----------------------------------------------------------------------------------------------------
  파생변수 생성 중... ✓
✓ 18개 파생변수 추가

🔤 Step 5: 범주형을 정수로 변환
----------------------------------------------------------------------------------------------------
  범주형 컬럼 감지: 10개
✓ 범주형을 정수로 변환 완료
✓ cat_features 인덱스: 10개
✓ 모든 컬럼 정수형으로 변환

🤖 Step 6: 25-Fold CatBoost (cat_features 유지)
---------------------------------------------------------------------------------

OSError: Cannot save file into a non-existent directory: 'data'

In [ ]:
import pandas as pd
import numpy as np

print("\n📤 Step 7: 제출 파일 생성 (경로 완벽 수정)")
print("-" * 100)

# test_predictions와 test_ids는 이미 위에서 생성됨
test_predictions = np.mean(test_pred_lst, axis=0)
test_predictions = np.clip(test_predictions, 0.001, 0.999)

# Colab 경로로 읽기
df_sub = pd.read_csv("/content/sample_submission.csv")

if 'ID' in df_sub.columns:
    df_sub['probability'] = test_predictions
else:
    df_sub = pd.DataFrame({
        'ID': test_ids,
        'probability': test_predictions
    })

# ⭐ Colab 경로로 저장!
OUTPUT_NAME = '/content/final_submission.csv'
df_sub.to_csv(OUTPUT_NAME, index=False)

print(f"✓ 제출 파일 생성 성공!")
print(f"  파일: {OUTPUT_NAME}")
print(f"  샘플: {len(df_sub):,}개")
print(f"  평균: {test_predictions.mean():.6f}")
print(f"  범위: [{test_predictions.min():.6f}, {test_predictions.max():.6f}]")

# 파일 확인
import os
files = os.listdir('/content/')
if 'final_submission.csv' in files:
    print("\n✅ final_submission.csv 생성 완료!")
    print("✅ 좌측 파일 메뉴에서 다운로드 가능!")
else:
    print("\n❌ 파일 생성 실패")


📤 Step 7: 제출 파일 생성 (경로 완벽 수정)
----------------------------------------------------------------------------------------------------
✓ 제출 파일 생성 성공!
  파일: /content/final_submission.csv
  샘플: 90,067개
  평균: 0.258868
  범위: [0.001000, 0.765619]

✅ final_submission.csv 생성 완료!
✅ 좌측 파일 메뉴에서 다운로드 가능!
